In [3]:
import pandas as pd
import numpy as np
from joblib import load

# Data baru
data_baru = pd.DataFrame({
    'namaproyek': ['Gedung Apartemen', 'Gedung Apartemen'],
    'waktu': [606, 722],
    'provinsi': ['DKI Jakarta', 'DKI Jakarta'],
    'tahun': [2021, 2023],
    'luas': [68204.25, 94550.57],
    'subitem': [6, 6],
    'tinggi': [120.02, 160.76],
    'lantai': [31, 41],
    'ikk': [121.42, 116.73],
    'ihbp': [109.64, 113.88],
    'inflasi': [1.53, 2.28]
})


# Transformasi log pada kolom numerikal
numerical_cols = ['waktu', 'tahun', 'luas', 'subitem', 'tinggi', 'lantai', 'ikk', 'ihbp', 'inflasi']
data_transformed = data_baru.copy()
data_transformed[numerical_cols] = np.log1p(data_transformed[numerical_cols])

# Muat encoder yang telah disimpan
encoder_namaproyek = load('../model/encoder_namaproyek.pkl')
encoder_provinsi = load('../model/encoder_provinsi.pkl')

# Transformasi kolom kategorikal
data_transformed['label_provinsi'] = encoder_provinsi.transform(data_transformed['provinsi'])
data_transformed['label_namaproyek'] = encoder_namaproyek.transform(data_transformed['namaproyek'])

# Muat scaler yang telah disimpan
scaler = load('../model/scaler.pkl')
data_transformed[numerical_cols] = scaler.transform(data_transformed[numerical_cols])

# Hapus kolom yang tidak diperlukan lagi
data_transformed.drop(columns=['lantai', 'namaproyek', 'provinsi'], inplace=True)

# Muat model Voting Regressor yang telah disimpan
voting_regressor2 = load('../model/voting_regressor2_model.joblib')

# Prediksi hasil menggunakan data baru
y_pred_baru = voting_regressor2.predict(data_transformed)

# Invers transformasi log (kembali ke nilai asli)
y_pred_asli = np.expm1(y_pred_baru)

# Buat DataFrame untuk hasil prediksi
hasil_prediksi = data_baru.copy()
hasil_prediksi['Hasil Prediksi'] = y_pred_asli

# Konversi hasil prediksi ke dalam bentuk Rupiah
hasil_prediksi['Hasil Prediksi (Rupiah)'] = hasil_prediksi['Hasil Prediksi'].apply(
    lambda x: f"Rp {x:,.2f}".replace(',', '.').replace('.', ',', 1)
)

# Drop kolom prediksi lama jika tidak diperlukan
hasil_prediksi.drop(columns=['Hasil Prediksi'], inplace=True)

# Tampilkan tabel hasil prediksi
hasil_prediksi

,namaproyek,waktu,provinsi,tahun,luas,subitem,tinggi,lantai,ikk,ihbp,inflasi,Hasil Prediksi (Rupiah)
0,Gedung Apartemen,606,DKI Jakarta,2021,68204.25,6,120.02,31,121.42,109.64,1.53,"Rp 870,624.103.339.17"
1,Gedung Apartemen,722,DKI Jakarta,2023,94550.57,6,160.76,41,116.73,113.88,2.28,"Rp 1,221.741.790.265.17"
